# Pad a batch to a length bucket

A `jit`-compiled function is recompiled for every new input shape. If you pad each batch to its own longest sequence, a serving loop sees dozens of lengths and spends its time compiling. The standard fix is to pad to a small set of **bucket lengths**, so only a few shapes ever reach the compiled function.

Write `bucket_batch(seqs, buckets, pad_id, left)`:

- `seqs`: a Python list of `B >= 1` one-dimensional int arrays (token ids). A sequence may be empty.
- `buckets`: a list of positive ints in ascending order.
- `pad_id`: int used for padding.
- `left`: bool. `False` pads on the right (tokens start at column 0). `True` pads on the left (tokens end at the last column), which is how batched decoding usually lines up prompts.

## Padded length
Let $m$ be the longest sequence length (0 if all are empty). The padded length $L$ is the smallest bucket with $L \ge m$. If $m$ is larger than the largest bucket $b_{\max}$, use the smallest multiple of $b_{\max}$ that is $\ge m$.

## Return
A tuple of three `(B, L)` arrays:

1. `tokens` (int32): the sequences placed as described, `pad_id` everywhere else.
2. `mask` (bool): `True` at real tokens.
3. `positions` (int32): position of each real token within its own sequence (`0, 1, 2, ...` from its first real token), and `0` at padding.

This is host-side preprocessing that runs before `jit`, so NumPy is fine; return NumPy or JAX arrays. This problem runs in **Colab** with the other JAX problems. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import numpy as np

def bucket_batch(seqs, buckets, pad_id, left):
    """Return (tokens int32 (B, L), mask bool (B, L), positions int32 (B, L))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [[{"__nd__": [5, 6, 7], "dtype": "int64"}, {"__nd__": [8], "dtype": "int64"}], [4, 8], 0, false], "e": {"__tup__": [{"__nd__": [[5, 6, 7, 0], [8, 0, 0, 0]], "dtype": "int32"}, {"__nd__": [[true, true, true, false], [true, false, false, false]], "dtype": "bool"}, {"__nd__": [[0, 1, 2, 0], [0, 0, 0, 0]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [5, 6, 7], "dtype": "int64"}, {"__nd__": [8], "dtype": "int64"}], [4, 8], 0, true], "e": {"__tup__": [{"__nd__": [[0, 5, 6, 7], [0, 0, 0, 8]], "dtype": "int32"}, {"__nd__": [[false, true, true, true], [false, false, false, true]], "dtype": "bool"}, {"__nd__": [[0, 0, 1, 2], [0, 0, 0, 0]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [1, 2, 3, 4], "dtype": "int64"}], [4, 8, 16], -1, false], "e": {"__tup__": [{"__nd__": [[1, 2, 3, 4]], "dtype": "int32"}, {"__nd__": [[true, true, true, true]], "dtype": "bool"}, {"__nd__": [[0, 1, 2, 3]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [], "dtype": "int64"}, {"__nd__": [3, 3], "dtype": "int64"}], [2, 4], 9, true], "e": {"__tup__": [{"__nd__": [[9, 9], [3, 3]], "dtype": "int32"}, {"__nd__": [[false, false], [true, true]], "dtype": "bool"}, {"__nd__": [[0, 0], [0, 1]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [], "dtype": "int64"}, {"__nd__": [], "dtype": "int64"}], [8, 16], 0, false], "e": {"__tup__": [{"__nd__": [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], "dtype": "int32"}, {"__nd__": [[false, false, false, false, false, false, false, false], [false, false, false, false, false, false, false, false]], "dtype": "bool"}, {"__nd__": [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [42, 13, 6, 15, 21, 40, 23, 5, 17, 30, 40], "dtype": "int64"}, {"__nd__": [36, 49, 10], "dtype": "int64"}], [4, 8], 0, true], "e": {"__tup__": [{"__nd__": [[0, 0, 0, 0, 0, 42, 13, 6, 15, 21, 40, 23, 5, 17, 30, 40], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 36, 49, 10]], "dtype": "int32"}, {"__nd__": [[false, false, false, false, false, true, true, true, true, true, true, true, true, true, true, true], [false, false, false, false, false, false, false, false, false, false, false, false, false, true, true, true]], "dtype": "bool"}, {"__nd__": [[0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [44, 3, 28, 14, 10], "dtype": "int64"}, {"__nd__": [33, 15, 28, 13, 8, 37, 22, 34, 33], "dtype": "int64"}, {"__nd__": [47, 21], "dtype": "int64"}, {"__nd__": [11, 32, 46, 48, 43, 34, 19], "dtype": "int64"}], [8, 16, 32], 0, false], "e": {"__tup__": [{"__nd__": [[44, 3, 28, 14, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [33, 15, 28, 13, 8, 37, 22, 34, 33, 0, 0, 0, 0, 0, 0, 0], [47, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [11, 32, 46, 48, 43, 34, 19, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "dtype": "int32"}, {"__nd__": [[true, true, true, true, true, false, false, false, false, false, false, false, false, false, false, false], [true, true, true, true, true, true, true, true, true, false, false, false, false, false, false, false], [true, true, false, false, false, false, false, false, false, false, false, false, false, false, false, false], [true, true, true, true, true, true, true, false, false, false, false, false, false, false, false, false]], "dtype": "bool"}, {"__nd__": [[0, 1, 2, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 2, 3, 4, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "dtype": "int32"}]}}, {"a": [[{"__nd__": [20, 2, 10, 17, 17, 29, 26, 34, 44, 43, 39, 48, 16, 45, 46, 11, 24], "dtype": "int64"}, {"__nd__": [28], "dtype": "int64"}], [16], 99, true], "e": {"__tup__": [{"__nd__": [[99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 20, 2, 10, 17, 17, 29, 26, 34, 44, 43, 39, 48, 16, 45, 46, 11, 24], [99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 28]], "dtype": "int32"}, {"__nd__": [[false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true, true], [false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, false, true]], "dtype": "bool"}, {"__nd__": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(bucket_batch, json.loads(_TESTS), 'jax', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")